# PAG bản 2 — chạy lại chuỗi dev (ảnh xoá ngoài phổi + thêm CheXFound)

**Vì sao chạy lại:** kiểm tra 50 ảnh thấy 18/50 còn sót chữ → tô đen mọi thứ ngoài bao lồi hai phổi. So sánh đặc trưng cũ/mới cho kết quả 2b (thay đổi đáng kể) theo luật đặt trước → chạy lại toàn bộ chuỗi dev. Đồng thời thêm bộ mã hoá ảnh (d) **CheXFound** làm cấu hình thứ hai (bộ chính vẫn là BioViL-T (a), không đổi sau khi xem kết quả).

**Thư mục:** code `scripts_v2/` · cache `cache_v2/` (chứa id → PII, KHÔNG đưa lên GitHub) · kết quả `results_v2/` · trọng số `chexfound/` (1,2 GB, không đưa vào repo). Kết quả cũ trong `results/` giữ nguyên để so sánh.

**Cách chạy:** chạy lần lượt từng ô. Mỗi ô gọi một script `.py` bằng subprocess, in tiến độ trực tiếp và ghi log vào `results_v2/logs/`. Ô nào đã có đầu ra thì tự bỏ qua (đặt `FORCE = True` để chạy lại). Các script chỉ in số liệu gộp, không in tên/tên tệp.

**Không chạy lại:** `v2_p0.py`, `v2_p05_check.py`, `v2_explore.py` (không phụ thuộc ảnh) và **test** (chỉ mở một lần sau khi thầy duyệt cấu hình cuối).

In [5]:
import os, sys, subprocess, time, shutil
from pathlib import Path

SCR = Path.cwd() if (Path.cwd() / "v2_lib.py").exists() else Path(r"E:\Hoc hanh\ĐÒ ÁN\Data DGCNN\New Merged DATA\PAG\scripts_v2")
PAG = SCR.parent
CACHE, RES, OLDCACHE = PAG / "cache_v2", PAG / "results_v2", PAG / "cache"
LOG = RES / "logs"; LOG.mkdir(parents=True, exist_ok=True)
FORCE = False      # True -> chay lai ca khi da co dau ra

assert (PAG / "chexfound" / "model.safetensors").exists(), "thieu trong so CheXFound"
assert (CACHE / "core.pkl").exists(), "thieu cache_v2/core.pkl (chia fold)"

def run(script, *args, out=None):
    """Chay `python -u script args` trong scripts_v2, in truc tiep + ghi log. out: tep dau ra -> bo qua neu da co."""
    tag = "_".join([Path(script).stem, *[a.strip("-") for a in args]])
    if out is not None and Path(out).exists() and not FORCE:
        print(f"[bo qua] {tag}: da co {Path(out).name}"); return
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1")
    t0 = time.time(); print(f"=== {tag} — bat dau {time.strftime('%H:%M:%S')}", flush=True)
    with open(LOG / f"{tag}.log", "w", encoding="utf-8") as lf:
        p = subprocess.Popen([sys.executable, "-u", script, *args], cwd=SCR, env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in p.stdout:
            print(line, end="", flush=True); lf.write(line)
        rc = p.wait()
    print(f"=== {tag} — {'XONG' if rc == 0 else f'LOI (ma {rc})'} sau {(time.time()-t0)/60:.1f} phut", flush=True)
    if rc: raise RuntimeError(f"{tag} loi, xem {LOG / (tag + '.log')}")

import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO")

GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## Bước 1 — Tiền xử lý ảnh + đặc trưng 4 bộ mã hoá (~35–45 phút)
Pad 1024 → che chữ → phân đoạn phổi → **tô đen ngoài bao lồi phổi** → cắt khung phổi → 512 → 6 vùng ILO → đặc trưng vùng cho a) BioViL-T, b) ResNet50-xrv, c) BioViL-T control, d) CheXFound. Ghi `cache_v2/img_v2.npz`.

In [6]:
run("v2_image.py", out=CACHE / "img_v2.npz")

[bo qua] v2_image: da co img_v2.npz


## Bước 2 — Báo cáo tiền xử lý + kiểm tra nhiễu lô chụp trên đặc trưng mới (P0.1 bước 5, P0.2)
In cho từng bộ mã hoá: ROC-AUC chỉ ảnh, AUC trong lô (tính trong từng lô chụp rồi lấy trung bình — loại phần “nhận ra lô”), Δ do ảnh so với chỉ biết lô, độ chính xác đoán lô.

In [7]:
run("v2_p01_after.py", out=RES / "tuan1_phase0" / "P0_1_nhieu_lo_chup_buoc5" / "BAO_CAO.md")

[bo qua] v2_p01_after: da co BAO_CAO.md


## Bước 3 — Mô hình đối chứng (L0, L1, Z1, L1-khám, L2, X0, X2, L3b, B3b, GAT)

In [8]:
run("v2_baselines.py", out=CACHE / "run_baselines.npz")

[bo qua] v2_baselines: da co run_baselines.npz


## Bước 4 — PG: dò 30 cấu hình trên validation nội bộ, 5 fold × 3 seed (lâu nhất)

In [9]:
run("v2_run.py", "PG", "--search", out=CACHE / "run_PG.npz")

=== v2_run_PG_search — bat dau 21:40:13
[PG] fold 0: cau hinh 6 epoch 5 | 264s
[PG] fold 1: cau hinh 21 epoch 11 | 540s
[PG] fold 2: cau hinh 1 epoch 13 | 921s
[PG] fold 3: cau hinh 6 epoch 7 | 1185s
[PG] fold 4: cau hinh 1 epoch 4 | 1486s
[PG] XONG 1486s | co anh AUC 0.764 | khong anh AUC 0.692 | trong lo 0.707
=== v2_run_PG_search — XONG sau 24.9 phut


## Bước 5 — T2 (Transformer bảng trên cùng node của PG, không cạnh) và T0 (chỉ lâm sàng)
T0 chỉ dùng lâm sàng, cùng chia fold → kết quả không đổi. Mặc định **chép** kết quả T0 cũ sang `cache_v2` để tiết kiệm thời gian; đặt `RERUN_T0 = True` nếu muốn chạy lại.

In [10]:
run("v2_run.py", "T2", "--search", out=CACHE / "run_T2.npz")

RERUN_T0 = False
if RERUN_T0:
    run("v2_run.py", "T0", "--search", out=CACHE / "run_T0.npz")
else:
    for f in ("run_T0.npz", "selected_T0.json", "log_T0.csv"):
        if (OLDCACHE / f).exists() and not (CACHE / f).exists(): shutil.copy2(OLDCACHE / f, CACHE / f)
    print("T0:", "da chep tu cache cu" if (CACHE / "run_T0.npz").exists() else "KHONG tim thay — dat RERUN_T0 = True")

=== v2_run_T2_search — bat dau 22:05:05
[T2] fold 0: cau hinh 23 epoch 21 | 2214s
[T2] fold 1: cau hinh 10 epoch 32 | 4679s
[T2] fold 2: cau hinh 20 epoch 31 | 7448s
[T2] fold 3: cau hinh 11 epoch 15 | 9635s
[T2] fold 4: cau hinh 9 epoch 28 | 12009s
[T2] XONG 12009s | co anh AUC 0.749 | khong anh AUC 0.684 | trong lo 0.678
=== v2_run_T2_search — XONG sau 200.3 phut
T0: da chep tu cache cu


## Bước 6 — Ablation (mỗi biến thể dùng cấu hình PG đã chọn ở từng fold)
`PG-enc-d` = PG với ảnh mã hoá bằng CheXFound (cấu hình thứ hai).

In [11]:
ABL = ["PG-noedge", "PG-complete", "PG-shuffle", "PG-global", "PG-noMD", "PG-noclin", "PG-noctx", "PG-RGCN",
       "PG-nobnn1", "PG-nobnn", "PG-enc-b", "PG-enc-c", "PG-enc-d"]
for n in ABL:
    run("v2_run.py", n, out=CACHE / f"run_{n}.npz")

=== v2_run_PG-noedge — bat dau 01:25:22
[PG-noedge] fold 0: cau hinh 6 epoch 5 | 22s
[PG-noedge] fold 1: cau hinh 21 epoch 9 | 43s
[PG-noedge] fold 2: cau hinh 1 epoch 5 | 58s
[PG-noedge] fold 3: cau hinh 6 epoch 6 | 77s
[PG-noedge] fold 4: cau hinh 1 epoch 7 | 94s
[PG-noedge] XONG 94s | co anh AUC 0.775 | khong anh AUC 0.669 | trong lo 0.722
=== v2_run_PG-noedge — XONG sau 1.7 phut
=== v2_run_PG-complete — bat dau 01:27:04
[PG-complete] fold 0: cau hinh 6 epoch 5 | 19s
[PG-complete] fold 1: cau hinh 21 epoch 12 | 45s
[PG-complete] fold 2: cau hinh 1 epoch 6 | 61s
[PG-complete] fold 3: cau hinh 6 epoch 7 | 81s
[PG-complete] fold 4: cau hinh 1 epoch 8 | 100s
[PG-complete] XONG 100s | co anh AUC 0.767 | khong anh AUC 0.690 | trong lo 0.708
=== v2_run_PG-complete — XONG sau 1.8 phut
=== v2_run_PG-shuffle — bat dau 01:28:49
[PG-shuffle] fold 0: cau hinh 6 epoch 5 | 20s
[PG-shuffle] fold 1: cau hinh 21 epoch 11 | 44s
[PG-shuffle] fold 2: cau hinh 1 epoch 13 | 72s
[PG-shuffle] fold 3: cau hi

## Bước 7 — Nhánh embedding (trích embedding PG → phân loại cổ điển + truy hồi ca tương tự)

In [12]:
run("v2_embed.py", out=CACHE / "emb_PG.npz")
run("v2_emb_eval.py", out=RES / "tuan5_nhanh_embedding" / "BAO_CAO.md")

=== v2_embed — bat dau 01:48:26
fold 0: dim 64 | AUC dau PG tren fold 0.730 | 6s
fold 1: dim 128 | AUC dau PG tren fold 0.754 | 12s
fold 2: dim 64 | AUC dau PG tren fold 0.704 | 19s
fold 3: dim 64 | AUC dau PG tren fold 0.766 | 24s
fold 4: dim 64 | AUC dau PG tren fold 0.773 | 27s
EMB XONG
=== v2_embed — XONG sau 0.6 phut
=== v2_emb_eval — bat dau 01:48:59
fold 0 xong
fold 1 xong
fold 2 xong
fold 3 xong
fold 4 xong
vector    bo_phan_loai      nhom  roc_auc            ci
     — đầu PG (3 seed)    có ảnh 0.763833 [0.734–0.791]
    E1              LR    có ảnh 0.747792 [0.716–0.773]
    E1             SVM    có ảnh 0.731039 [0.702–0.759]
    E1             kNN    có ảnh 0.753334 [0.719–0.779]
   tho              LR    có ảnh 0.747135 [0.717–0.777]
   tho             SVM    có ảnh 0.668573 [0.635–0.699]
   tho             kNN    có ảnh 0.655714 [0.623–0.684]
     — đầu PG (3 seed) không ảnh 0.691673 [0.666–0.715]
    E1              LR không ảnh 0.684355 [0.659–0.710]
    E1             SV

## Bước 8 — Báo cáo dev (tuần 2–5) + bảng tổng hợp
Luôn chạy lại (nhanh). **Không** có `--test`.

In [13]:
run("v2_report.py")
run("v2_extras.py")

=== v2_report — bat dau 01:52:02
BAO CAO XONG
=== v2_report — XONG sau 4.0 phut
=== v2_extras — bat dau 01:56:04
           python                             3.13.2
     he_dieu_hanh          Windows-11-10.0.26300-SP0
            numpy                              2.2.2
           pandas                              2.2.3
          sklearn                              1.7.2
            scipy                             1.16.3
            torch                        2.6.0+cu124
          xgboost                              3.4.1
  torchxrayvision                              1.5.5
              cv2                             4.13.0
health_multimodal                              0.2.2
  torch_geometric                        2.8.0.post1
             cuda                               12.4
              gpu NVIDIA GeForce RTX 3060 Laptop GPU
                          doi_tuong        sha256_16  so_hang    dev   test  so_nguoi_dev
phân chia id → (fold2, dev2, split) 10439d3d461fe4e2   

## Xem kết quả
Giả thuyết chính (H1a: PG không kém L1 ở người có phim; H1b: PG không kém L0 ở người không phim; biên −0,02 ROC-AUC) và ablation.

In [14]:
from IPython.display import Markdown, display
for d in ("tuan4_so_sanh_chinh", "tuan5_ablation", "tuan1_phase0/P0_1_nhieu_lo_chup_buoc5"):
    f = RES / d / "BAO_CAO.md"
    display(Markdown(f.read_text(encoding="utf-8") if f.exists() else f"*chưa có {d}*"))

# Tuần 4 — Kiểm định các giả thuyết khai báo trước (dev OOF)

Khai báo trước: `gia_thuyet_khai_bao_truoc_v2.json`. Biên không kém = −0,02 AUC. Holm (một phía, α = 0,025) trên 4 so sánh chính, dùng tỉ lệ lần bootstrap có Δ ≤ −0,02 làm đại lượng sắp thứ tự. H2 (PG vs PG-E6shuffle) và H4 (chú ý vs vùng tổn thương) KHÔNG đánh giá được.

### Bốn so sánh chính

| ma | so_sanh | nhom | n | auc_a | auc_b | delta | ci | P_delta_duong | ti_le_duoi_bien | dat_theo_KTC | p_Holm | dat_sau_Holm | delta_trong_lo | ci_trong_lo |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| H1a | PG vs L1 | có ảnh | 1500 | 0.756 | 0.754 | 0.003 | [-0.018–0.021] | 0.573 | 0.015 | True | 0.030 | False | -0.006 | [-0.032–0.018] |
| H1b | PG vs L0 | không ảnh | 5183 | 0.674 | 0.708 | -0.034 | [-0.053–-0.014] | 0.000 | 0.926 | False | 0.926 | False | nan |  |
| H3a | PG vs T2 | có ảnh | 1500 | 0.756 | 0.748 | 0.009 | [-0.014–0.031] | 0.774 | 0.006 | True | 0.018 | True | nan |  |
| H3b | PG vs T2 | không ảnh | 5183 | 0.674 | 0.658 | 0.016 | [-0.004–0.034] | 0.938 | 0.000 | True | 0.000 | True | nan |  |

_Tệp: `gia_thuyet_chinh.csv`_

*Δ = AUC(PG) − AUC(đối chứng), ghép cặp trên cùng lượt khám. `ci` = KTC 95% bootstrap theo người. `P_delta_duong` = tỉ lệ lần bootstrap Δ>0 (mức chắc chắn PG tốt hơn; KHÔNG phải p-value). `ti_le_duoi_bien` = tỉ lệ lần bootstrap Δ ≤ −0,02 (càng nhỏ càng chắc PG không kém). `dat_theo_KTC`: cận dưới KTC > −0,02. `p_Holm`: tỉ lệ bootstrap dưới biên sau hiệu chỉnh Holm (dùng như p một phía cho giả thuyết “PG kém hơn quá 0,02”; đạt khi ≤ 0,025). `dat_sau_Holm`: vẫn đạt sau hiệu chỉnh đa so sánh Holm. `delta_trong_lo`: Δ tính bằng AUC trong lô chụp (H1a theo PDF yêu cầu báo cả chỉ số này).*

**Điểm dừng:** H1 KHÔNG đạt ở cả hai nhóm; H2 không đánh giá được → theo luật khai báo trước (H1 và H2 cùng không đạt) cần dừng và xem lại thiết kế trước khi đi tiếp.

### So sánh khám phá (không hiệu chỉnh đa so sánh, chỉ để tham khảo)

| so_sanh | nhom | n | auc_a | auc_b | delta | ci | P_delta_duong |
|---|---|---|---|---|---|---|---|
| PG vs PG-shuffle | có ảnh | 1500 | 0.756 | 0.755 | 0.002 | [-0.001–0.004] | 0.931 |
| PG vs PG-noedge | có ảnh | 1500 | 0.756 | 0.747 | 0.010 | [-0.003–0.022] | 0.938 |
| PG vs L2 | có ảnh | 1500 | 0.756 | 0.725 | 0.031 | [0.010–0.052] | 0.998 |
| PG vs L2 | không ảnh | 5183 | 0.674 | 0.705 | -0.031 | [-0.051–-0.012] | 0.001 |
| PG vs GAT | có ảnh | 1500 | 0.756 | 0.716 | 0.041 | [0.020–0.061] | 1.000 |
| PG vs GAT | không ảnh | 5183 | 0.674 | 0.673 | 0.000 | [-0.018–0.018] | 0.505 |
| PG vs B3b | không ảnh | 5183 | 0.674 | 0.696 | -0.022 | [-0.044–0.001] | 0.033 |

_Tệp: `kham_pha.csv`_


# Tuần 5 — Ablation: mỗi lần bỏ/đổi một thành phần của PG

### Ablation — dev OOF đã hiệu chỉnh

| bien_the | mo_ta | auc_có ảnh | auc_trong_lo | auc_không ảnh | delta_có ảnh | ci_có ảnh | delta_không ảnh | ci_không ảnh |
|---|---|---|---|---|---|---|---|---|
| PG | đầy đủ | 0.756 | 0.704 | 0.674 | nan | nan | nan | nan |
| PG-drop | PG bình thường, bỏ node ảnh lúc đánh giá | 0.658 | 0.608 | 0.678 | -0.098 | [-0.128–-0.071] | 0.004 | [-0.001–0.008] |
| PG-noedge | bỏ hết cạnh (chỉ tự nối + CLS) | 0.747 | 0.682 | 0.640 | -0.010 | [-0.022–0.003] | -0.034 | [-0.049–-0.018] |
| PG-complete | đồ thị đầy đủ, một loại cạnh | 0.770 | 0.715 | 0.676 | 0.013 | [0.000–0.026] | 0.003 | [-0.008–0.014] |
| PG-shuffle | xáo loại cạnh giữa các vùng (phá giải phẫu) | 0.755 | 0.703 | 0.672 | -0.002 | [-0.004–0.001] | -0.002 | [-0.004–0.001] |
| PG-global | 1 node ảnh toàn cục thay 6 vùng | 0.723 | 0.671 | 0.667 | -0.033 | [-0.055–-0.012] | -0.007 | [-0.022–0.009] |
| PG-noMD | không modality dropout | 0.727 | 0.665 | 0.682 | -0.029 | [-0.052–-0.006] | 0.009 | [-0.009–0.026] |
| PG-noclin | chỉ ảnh + khám (bỏ 5 nhóm lâm sàng, người có phim) | 0.762 | 0.721 | nan | 0.006 | [-0.013–0.024] | nan | nan |
| PG-noctx | bỏ node tỉnh | 0.760 | 0.710 | 0.641 | 0.004 | [-0.005–0.014] | -0.033 | [-0.043–-0.022] |
| PG-RGCN | R-GCN thay Graph Transformer | 0.755 | 0.703 | 0.638 | -0.002 | [-0.014–0.010] | -0.036 | [-0.051–-0.019] |
| PG-nobnn1 | chỉ bỏ biến bnn | 0.758 | 0.704 | 0.659 | 0.002 | [-0.009–0.013] | -0.015 | [-0.026–-0.003] |
| PG-nobnn | bỏ bnn + 4 biến tiền sử bệnh (tiensuhh, benhhh, tsnoi, tsngoai) | 0.746 | 0.690 | 0.671 | -0.011 | [-0.026–0.006] | -0.002 | [-0.020–0.016] |
| PG-enc-b | ảnh bằng ResNet50-xrv | 0.770 | 0.723 | 0.654 | 0.014 | [-0.008–0.036] | -0.020 | [-0.036–-0.004] |
| PG-enc-c | ảnh bằng BioViL-T control | 0.722 | 0.668 | 0.645 | -0.035 | [-0.055–-0.014] | -0.028 | [-0.045–-0.011] |
| PG-enc-d | ảnh bằng CheXFound | 0.774 | 0.720 | 0.651 | 0.017 | [-0.005–0.040] | -0.023 | [-0.039–-0.006] |

_Tệp: `ablation.csv`_

*`delta_*` = AUC(biến thể) − AUC(PG) ghép cặp, kèm KTC 95%; âm nghĩa là bỏ thành phần đó làm mô hình kém đi. PG-E6shuffle / PG-noE6 / PG-1exam trùng PG trong cấu trúc dự phòng nên không chạy.*


# P0.1 bước 5 — Nhiễu lô chụp SAU che chữ + cắt theo phổi

### Đoán lô (18 lớp) từ đặc trưng ảnh đã che chữ + cắt

| dac_trung | do_chinh_xac_doan_lo | muc_doan_bua |
|---|---|---|
| BioViL-T đóng băng — 6 vùng | 0.408 | 0.111 |
| BioViL-T đóng băng — toàn ảnh | 0.420 | 0.111 |
| BioViL-T control — 6 vùng | 0.372 | 0.111 |
| BioViL-T control — toàn ảnh | 0.406 | 0.111 |
| ResNet50-xrv — 6 vùng | 0.404 | 0.111 |
| ResNet50-xrv — toàn ảnh | 0.404 | 0.111 |
| CheXFound — 6 vùng | 0.494 | 0.111 |
| CheXFound — toàn ảnh | 0.555 | 0.111 |

_Tệp: `doan_lo_tu_anh.csv`_

*Độ chính xác đoán lô càng gần `muc_doan_bua` (luôn đoán lô đông nhất) thì đặc trưng càng ít mang dấu vết của lô.*

### AUC gộp, AUC trong lô, Δ do ảnh so với chỉ biết lô, và fold để nguyên lô

| dac_trung | L1_auc | L1_auc_ci | auc_trong_lo | auc_trong_lo_ci | auc_chi_lo | delta_anh | delta_ci | auc_de_nguyen_lo |
|---|---|---|---|---|---|---|---|---|
| BioViL-T đóng băng — 6 vùng | 0.738 | [0.710–0.768] | 0.695 | [0.658–0.734] | 0.707 | 0.055 | [0.024–0.087] | 0.740 |
| BioViL-T đóng băng — toàn ảnh | 0.767 | [0.740–0.793] | 0.729 | [0.696–0.763] | 0.707 | 0.086 | [0.058–0.114] | 0.761 |
| BioViL-T control — 6 vùng | 0.721 | [0.693–0.749] | 0.694 | [0.661–0.725] | 0.707 | 0.049 | [0.019–0.081] | 0.688 |
| BioViL-T control — toàn ảnh | 0.759 | [0.733–0.785] | 0.725 | [0.692–0.754] | 0.707 | 0.085 | [0.060–0.110] | 0.742 |
| ResNet50-xrv — 6 vùng | 0.700 | [0.671–0.731] | 0.676 | [0.639–0.710] | 0.707 | 0.023 | [-0.011–0.059] | 0.685 |
| ResNet50-xrv — toàn ảnh | 0.753 | [0.725–0.779] | 0.732 | [0.696–0.765] | 0.707 | 0.077 | [0.046–0.108] | 0.737 |
| CheXFound — 6 vùng | 0.768 | [0.742–0.792] | 0.726 | [0.682–0.766] | 0.707 | 0.074 | [0.041–0.106] | 0.753 |
| CheXFound — toàn ảnh | 0.746 | [0.719–0.775] | 0.710 | [0.676–0.746] | 0.707 | 0.070 | [0.038–0.102] | 0.732 |

_Tệp: `ket_qua.csv`_

*`L1_auc`: ROC-AUC (xác suất mô hình xếp một ca dương cao hơn một ca âm; 0.5 = đoán bừa) của hồi quy logistic chỉ trên ảnh. `auc_trong_lo`: AUC tính riêng trong từng lô chụp rồi lấy trung bình theo cỡ lô; nó loại phần mô hình chỉ “nhận ra lô”. `delta_anh`: AUC(lô + ảnh) − AUC(chỉ lô); dương nghĩa là ảnh thêm thông tin ngoài lô. `auc_de_nguyen_lo`: mỗi lô nằm trọn trong một fold; mô hình phải dự đoán cho lô nó chưa thấy. CI = khoảng tin cậy 95% bootstrap theo người.*

### Để so sánh: kết quả bước 1–4 (đặc trưng cũ, chưa che chữ)

| dac_trung_truoc_che (toàn ảnh, CenterCrop) | auc_trong_lo_truoc | delta_truoc |
|---|---|---|
| BioViL-T frozen (toàn ảnh) | 0.722 | 0.077 |
| BioViL-T control (toàn ảnh) | 0.729 | 0.088 |
| ResNet50-xrv (toàn ảnh, bản 1) | 0.701 | 0.061 |

_Tệp: `truoc_che_de_so_sanh.csv`_

### Quyết định cuối P0.1 theo luật §2 (≥0.70 và Δ>0 → đi tiếp; 0.60–0.70 → có điều kiện; <0.60 → dừng)

| dac_trung | auc_trong_lo | delta_ci_duoi | quyet_dinh |
|---|---|---|---|
| BioViL-T đóng băng — 6 vùng | 0.695 | 0.024 | ĐI TIẾP CÓ ĐIỀU KIỆN — che chữ + cắt bắt buộc |
| BioViL-T đóng băng — toàn ảnh | 0.729 | 0.058 | ĐI TIẾP — AUC trong lô thành chỉ số chính |
| BioViL-T control — 6 vùng | 0.694 | 0.019 | ĐI TIẾP CÓ ĐIỀU KIỆN — che chữ + cắt bắt buộc |
| BioViL-T control — toàn ảnh | 0.725 | 0.060 | ĐI TIẾP — AUC trong lô thành chỉ số chính |
| ResNet50-xrv — 6 vùng | 0.676 | -0.011 | ĐI TIẾP CÓ ĐIỀU KIỆN — che chữ + cắt bắt buộc |
| ResNet50-xrv — toàn ảnh | 0.732 | 0.046 | ĐI TIẾP — AUC trong lô thành chỉ số chính |
| CheXFound — 6 vùng | 0.726 | 0.041 | ĐI TIẾP — AUC trong lô thành chỉ số chính |
| CheXFound — toàn ảnh | 0.710 | 0.038 | ĐI TIẾP — AUC trong lô thành chỉ số chính |

_Tệp: `quyet_dinh.csv`_


### So sánh ROC-AUC bản cũ (`results/`) và bản 2 (`results_v2/`)
ROC-AUC = xác suất mô hình xếp một ca dương cao hơn một ca âm (0,5 = đoán bừa). AUC trong lô = AUC tính trong từng lô chụp rồi lấy trung bình.

In [15]:
import pandas as pd
cols = ["có ảnh | roc_auc", "có ảnh | auc_trong_lo", "không ảnh | roc_auc", "toàn bộ | roc_auc"]
old = pd.read_csv(PAG / "results" / "tong_hop" / "bang_tong_hop.csv").set_index("mo_hinh")[cols]
new = pd.read_csv(RES / "tong_hop" / "bang_tong_hop.csv").set_index("mo_hinh")[cols]
cmp = pd.concat({"cu": old, "moi": new}, axis=1).swaplevel(axis=1).sort_index(axis=1, level=0)
cmp.round(3)

có ảnh | auc_trong_lo        có ảnh | roc_auc         \
                           cu    moi               cu    moi   
mo_hinh                                                        
B3b                     0.630  0.638            0.701  0.702   
GAT                     0.629  0.652            0.704  0.716   
L0                      0.554  0.554            0.627  0.627   
L1                      0.689  0.710            0.741  0.754   
L1-khám                 0.455  0.455            0.481  0.481   
L2                      0.644  0.659            0.711  0.725   
L3b                     0.643  0.658            0.711  0.724   
PG                      0.695  0.704            0.754  0.756   
T0                      0.589  0.589            0.652  0.652   
T2                      0.703  0.689            0.761  0.748   
X0                      0.579  0.579            0.648  0.648   
X2                      0.654  0.666            0.713  0.726   
Z1                      0.625  0.652            0.690  0.704   

        không ảnh | roc_auc        toàn bộ | roc_auc         
                         cu    moi                cu    moi  
mo_hinh                                                      
B3b                   0.680  0.696             0.738  0.745  
GAT                   0.691  0.673             0.743  0.737  
L0                    0.708  0.708             0.736  0.736  
L1                      NaN    NaN             0.741  0.754  
L1-khám               0.485  0.485             0.626  0.626  
L2                    0.707  0.705             0.751  0.752  
L3b                   0.707  0.706             0.751  0.753  
PG                    0.679  0.674             0.747  0.744  
T0                    0.639  0.639             0.707  0.707  
T2                    0.664  0.658             0.737  0.730  
X0                    0.709  0.709             0.742  0.742  
X2                    0.714  0.712             0.759  0.758  
Z1                      NaN    NaN             0.690  0.704